# Subject 4: Prediction of the Critical Temperature of Superconducting Materials

**Authors:** Xiaopeng Zhang, Marçal Herraiz Bayó, Shuaibo HUANG,
Carlos Cosentino, Polina Ptukha, and Lyes Bouchoucha.

This notebook is the clean code submission for the statistical learning project.
The objective is to predict the critical temperature `critical_temp` from 81
physical and chemical covariates in the UCI superconductivity dataset.

In [ ]:
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

## Reproducible Pipeline

The notebook delegates implementation to reusable Python modules and scripts.
This avoids duplicating notebook-only logic and keeps preprocessing, model
selection, and evaluation reproducible.

In [ ]:
for command in [
    [sys.executable, "scripts/run_eda.py"],
    [sys.executable, "scripts/train_models.py"],
    [sys.executable, "scripts/evaluate_models.py"],
]:
    print("$", " ".join(command))
    subprocess.run(command, check=True, cwd=PROJECT_ROOT)

## Dataset Overview

In [ ]:
preprocessing = pd.read_json(
    PROJECT_ROOT / "reports/tables/preprocessing_summary.json",
    typ="series",
)
overview = pd.read_json(
    PROJECT_ROOT / "reports/tables/dataset_overview.json",
    typ="series",
)
target_summary = pd.read_json(
    PROJECT_ROOT / "reports/tables/target_summary.json",
    typ="series",
)
display(preprocessing.to_frame("value"))
display(overview.to_frame("value"))
display(target_summary.to_frame("value"))

elements = pd.read_csv(PROJECT_ROOT / "reports/tables/target_by_number_of_elements.csv")
display(elements)

The raw table contains exact duplicate rows, which are removed before the
train-test split. The processed table has no missing values. The target is positive
and right-skewed, so the modeling pipeline uses a `log1p` target transformation and
reports final errors after transforming predictions back to Kelvin. The grouped
summary by `number_of_elements` shows that compositions with more elements are much
more likely to include high critical-temperature observations.

In [ ]:
for figure in [
    "reports/figures/target_distribution.png",
    "reports/figures/log_target_distribution.png",
    "reports/figures/target_by_number_of_elements.png",
]:
    display(Image(filename=str(PROJECT_ROOT / figure)))

## Exploratory Feature Analysis

In [ ]:
correlations = pd.read_csv(
    PROJECT_ROOT / "reports/tables/feature_target_correlations.csv"
)
family_correlations = pd.read_csv(
    PROJECT_ROOT / "reports/tables/feature_family_correlation_summary.csv"
)
display(correlations.head(15))
display(family_correlations)

for figure in [
    "reports/figures/feature_target_correlations.png",
    "reports/figures/selected_feature_correlation_matrix.png",
    "reports/figures/feature_family_correlations.png",
]:
    display(Image(filename=str(PROJECT_ROOT / figure)))

## Model Comparison

The final comparison below is computed only on the held-out test set. Hyperparameters
are selected inside the training data by cross-validation where applicable.

In [ ]:
comparison = pd.read_csv(PROJECT_ROOT / "reports/tables/final_model_comparison.csv")
display(comparison)

full_comparison = pd.read_csv(PROJECT_ROOT / "reports/tables/model_comparison.csv")
display(full_comparison)

## Diagnosis of the Recommended Model

The recommended model is the one with the smallest test RMSE. Residual plots are used
to check whether the error pattern reveals major systematic failures. We also
summarize the selected model by observed target range because the high-temperature
tail is scientifically important and harder to predict.

In [ ]:
residual_summary = pd.read_csv(
    PROJECT_ROOT / "reports/tables/final_model_residuals_summary.csv"
)
range_errors = pd.read_csv(
    PROJECT_ROOT / "reports/tables/best_model_error_by_target_range.csv"
)
display(residual_summary)
display(range_errors)

In [ ]:
for figure in [
    "reports/figures/best_model_predicted_vs_observed.png",
    "reports/figures/best_model_residual_diagnostics.png",
    "reports/figures/best_model_error_by_target_range.png",
]:
    display(Image(filename=str(PROJECT_ROOT / figure)))

## Conclusion

On the cleaned table, k-NN regression gives the lowest held-out RMSE in this run,
narrowly ahead of `HistGradientBoostingRegressor` and clearly ahead of OLS, Ridge,
and Lasso. Ridge, Lasso, k-NN, and the general evaluation framework are covered by
the course material; the gradient boosting model is included as one additional
nonlinear benchmark and must be explained explicitly in the written report.